<a href="https://colab.research.google.com/github/AymaneAmri-MD/project-23-link/blob/main/notebooks/01_binomial_vs_gaussian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Binomial vs Gaussian: the ideal $(3+1)^n$ model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USER/project-23-link/blob/main/notebooks/01_binomial_vs_gaussian.ipynb)

For $n$ independent loci with dominance probability $p=0.75$ (an $Aa \times Aa$ cross), the number of dominant phenotypes is $X \sim \mathrm{Binomial}(n, p)$. This notebook checks the numbers of the original note and measures how good the Gaussian approximation really is.

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/AymaneAmri-MD/project-23-link.git"

def find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "src" / "link23").exists():
            return q
    return None

ROOT = find_root()
if ROOT is None:  # e.g. Google Colab: clone the repo first
    subprocess.run(["git", "clone", REPO_URL, "project-23-link"], check=True)
    ROOT = Path("project-23-link").resolve()
sys.path.insert(0, str(ROOT / "src"))
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from link23 import *
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
rng = np.random.default_rng(23)

ModuleNotFoundError: No module named 'link23'

## 1. Numbers from the note

In [ ]:
n, p = 23, 0.75
mu, sigma = n * p, np.sqrt(n * p * (1 - p))
print(f"mu    = {mu:.2f}")
print(f"sigma = {sigma:.4f}   (the note rounds to 2.07; 2.08 is the correct rounding)")
print(f"P(X=0) = 0.25^23 = {0.25**23:.3e} = 2^-46 = {2.0**-46:.3e}")
print(f"P(X=23) = 0.75^23 = {0.75**23:.4f}")

## 2. From $n=1$ to $n=23$

In [ ]:
ns = [1, 2, 5, 10, 23]
fig, axes = plt.subplots(1, len(ns), figsize=(15, 3), sharey=False)
for ax, n_ in zip(axes, ns):
    k = np.arange(n_ + 1)
    ax.bar(k, binomial_pmf(n_), color="#4C72B0", alpha=.8)
    x = np.linspace(-0.5, n_ + .5, 300)
    ax.plot(x, stats.norm.pdf(x, n_ * .75, np.sqrt(n_ * .1875)), "r-", lw=1.5)
    ax.set_title(f"n = {n_}"); ax.set_xlabel("k dominant")
axes[0].set_ylabel("P(X = k)")
plt.suptitle("Binomial (bars) vs Gaussian approximation (red)", y=1.05)
plt.tight_layout(); plt.savefig(FIG / "01_binomial_vs_gaussian.png", bbox_inches="tight"); plt.show()

## 3. How good is the Gaussian? Total variation distance and tails

The distribution is skewed (skewness $=(1-2p)/\sqrt{np(1-p)}$), so convergence is slow. The tails are where the approximation fails most: the Gaussian badly misestimates $P(X=0)$.

In [ ]:
rows = []
for n_ in range(1, 61):
    k = np.arange(n_ + 1)
    b = binomial_pmf(n_)
    g = stats.norm.cdf(k + .5, n_ * .75, np.sqrt(n_ * .1875)) - stats.norm.cdf(k - .5, n_ * .75, np.sqrt(n_ * .1875))
    rows.append((n_, total_variation(b, g / g.sum())))
rows = np.array(rows)
plt.figure(figsize=(5.5, 3.2))
plt.plot(rows[:, 0], rows[:, 1], "o-", ms=3)
plt.axvline(23, color="gray", ls=":")
plt.xlabel("n (number of loci)"); plt.ylabel("TV distance to Gaussian")
plt.title("Convergence of Binomial(n, 0.75) to its Gaussian")
plt.tight_layout(); plt.savefig(FIG / "01_tv_distance.png", bbox_inches="tight"); plt.show()

gauss_p0 = stats.norm.cdf(0.5, 17.25, sigma)
print(f"Exact  P(X=0) = {0.25**23:.2e}")
print(f"Gauss  P(X<=0.5) = {gauss_p0:.2e}  -> off by a factor {gauss_p0 / 0.25**23:.1e}")

## 4. Monte Carlo check

In [ ]:
counts = rng.binomial(23, 0.75, size=1_000_000)
print("simulated mean/var:", counts.mean().round(3), counts.var().round(3), "| theory:", 17.25, 4.3125)
print("P(X=0) never observed in 1e6 draws:", (counts == 0).sum() == 0)

## 5. Dominance counting is not an additive polygenic trait

The note counts *dominant phenotypes over 23 independent Mendelian traits*. A genuine additive polygenic trait sums allele effects over $2n$ alleles: with $n$ biallelic loci in Hardy-Weinberg at frequency $0.5$ it follows Binomial$(2n, 0.5)$. The two models have different means, variances and shapes.

In [ ]:
n_ = 23
k = np.arange(2 * n_ + 1)
plt.figure(figsize=(6.5, 3.2))
plt.bar(np.arange(n_ + 1), binomial_pmf(n_), alpha=.6, label="# dominant phenotypes: Bin(23, 0.75)")
plt.bar(k, stats.binom.pmf(k, 2 * n_, .5), alpha=.5, label="additive allele count: Bin(46, 0.5)")
plt.xlabel("score"); plt.ylabel("probability"); plt.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "01_dominance_vs_additive.png", bbox_inches="tight"); plt.show()

**Take-away.** The binomial note is correct as a *toy* model; calling it 'polygenic' requires an additive model, and the Gaussian is a poor description of the extreme tails. Next: what linkage does to this distribution (notebook 02).